In [ ]:
# =========================================================
# 1. provide core functions for log-odds computation
# 2. compute log-odds of outcome levels by context levels
# =========================================================
import math
import pandas as pd
import numpy as np
from scipy import stats
from typing import Callable, Optional, Dict, Any, Tuple, Union, Literal

import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))

from stats.filtering import apply_filters, FilterValue, has_value,  _topn_values

# =========================================================
# Core stats
# =========================================================
def log_odds_2x2(
    a: float, b: float, c: float, d: float,
    alpha: float = 0.5,
    add_alpha_if_zero: bool = True
) -> Tuple[float, float, float, float, float]:
    """
    Returns (odds_ratio, log_or, se, z, p_value)
    If any cell is 0 and add_alpha_if_zero=True, adds alpha to ALL four cells.
    """
    if add_alpha_if_zero and (a == 0 or b == 0 or c == 0 or d == 0):
        a, b, c, d = a + alpha, b + alpha, c + alpha, d + alpha

    odds_ratio = (a * d) / (b * c)
    log_or = math.log(odds_ratio)
    se = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    z = log_or / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return odds_ratio, log_or, se, z, p

# =========================================================
# Main: context × outcome (multi-level outcome supported)
# =========================================================
def compute_logodds_by_context_by_outcome(
    df: pd.DataFrame,
    *,
    context_col: str,                                   # e.g., "category"
    outcome_col: str,                                   # e.g., "EP_form" or "EP_T"
    outcome_transform: Optional[Callable[[Any], Any]] = None,
    # If outcome is boolean already, keep transform None.
    # If outcome needs cleaning (strip tags, map labels...), set this.

    # Counting mode
    count_mode: Literal["auto", "rows", "weight"] = "auto",
    weight_col: str = "ID",

    # Filters
    filters: Optional[Dict[str, Any]] = None,

    # Limit contexts
    context_top_n: Optional[int] = None,
    context_min_count: Optional[float] = None,

    # Limit outcome levels (IMPORTANT for long outcome lists)
    outcome_top_n: Optional[int] = None,          # default: keep top 30 outcomes to avoid explosion
    outcome_min_count: Optional[float] = None,  # e.g., 100 (weight or rows depending on count_mode)

    # Log-odds settings
    alpha: float = 0.5,
    add_alpha_if_zero: bool = True,

    # Optional global association test: context × outcome table
    do_chi2: bool = True
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Computes log-odds for each (context g, outcome level o) using one-vs-rest.

    For each context g and outcome level o:
      a = count(context=g AND outcome=o)
      b = count(context=g AND outcome!=o)
      c = count(context!=g AND outcome=o)
      d = count(context!=g AND outcome!=o)

    Returns:
      result_df: one row per (g, o)
      chi2_df: 1-row summary for overall context×outcome association (optional)
    """
    if count_mode == "auto":
        count_mode = "weight" if weight_col else "rows"
        
    needed = {context_col, outcome_col}
    if count_mode == "weight":
        needed.add(weight_col)

    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    f = df.copy()
    if filters:
        f = apply_filters(f, filters=filters)

    # normalize outcome if needed
    if outcome_transform is not None:
        f["_OUTCOME_VAL"] = f[outcome_col].map(outcome_transform)
    else:
        f["_OUTCOME_VAL"] = f[outcome_col]

    # choose contexts/outcomes by frequency consistent with count_mode
    context_values = _topn_values(
        f, context_col,
        mode=count_mode, weight_col=weight_col,
        top_n=context_top_n, min_count=context_min_count, dropna=True
    )

    outcome_values = _topn_values(
        f, "_OUTCOME_VAL",
        mode=count_mode, weight_col=weight_col,
        top_n=outcome_top_n, min_count=outcome_min_count, dropna=True
    )

    f = f[f[context_col].isin(context_values) & f["_OUTCOME_VAL"].isin(outcome_values)].copy()

    # build contingency counts: context × outcome
    if count_mode == "weight":
        tab = (
            f.groupby([context_col, "_OUTCOME_VAL"], dropna=False)[weight_col]
             .sum()
             .unstack("_OUTCOME_VAL", fill_value=0.0)
        )
    else:
        tab = (
            f.groupby([context_col, "_OUTCOME_VAL"], dropna=False)
             .size()
             .unstack("_OUTCOME_VAL", fill_value=0.0)
        )

    # ensure all selected contexts/outcomes exist
    tab = tab.reindex(index=context_values, columns=outcome_values, fill_value=0.0)

    # totals
    context_totals = tab.sum(axis=1)     # per context g
    outcome_totals = tab.sum(axis=0)     # per outcome o
    grand_total = float(tab.values.sum())

    # compute per (g, o) 2x2 and log-odds
    rows = []
    for g in context_values:
        g_total = float(context_totals.loc[g])

        for o in outcome_values:
            a = float(tab.loc[g, o])
            b = float(g_total - a)
            c = float(outcome_totals.loc[o] - a)
            d = float(grand_total - a - b - c)

            odds_ratio, log_or, se, z, p = log_odds_2x2(
                a, b, c, d,
                alpha=alpha,
                add_alpha_if_zero=add_alpha_if_zero
            )

            rows.append({
                context_col: g,
                "outcome_level": o,

                "a_in_context_and_level": a,
                "b_in_context_not_level": b,
                "c_not_context_in_level": c,
                "d_not_context_not_level": d,

                "odds_ratio": odds_ratio,
                "log_odds": log_or,
                "SE": se,
                "z": z,
                "p_value": p,

                "context_total": g_total,
                "outcome_total": float(outcome_totals.loc[o]),
                "grand_total": grand_total,

                "count_mode": count_mode,
                "alpha_added": (alpha if (add_alpha_if_zero and (a == 0 or b == 0 or c == 0 or d == 0)) else 0.0),
            })

    result_df = pd.DataFrame(rows)

    # optional overall chi-square on the full contingency table
    chi2_df = None
    if do_chi2:
        table = tab.values.astype(float)

        # drop empty rows/cols (shouldn't happen often after filtering, but safe)
        row_sum = table.sum(axis=1)
        col_sum = table.sum(axis=0)
        keep_r = row_sum > 0
        keep_c = col_sum > 0
        table_valid = table[np.ix_(keep_r, keep_c)]

        if table_valid.shape[0] >= 2 and table_valid.shape[1] >= 2 and table_valid.sum() > 0:
            try:
                chi2, p_ctx, dfree, _ = stats.chi2_contingency(table_valid)

                # Cramér’s V for RxC
                n = float(table_valid.sum())
                r, c_ = table_valid.shape
                denom = n * (min(r - 1, c_ - 1))
                cramers_v = math.sqrt(float(chi2) / denom) if denom > 0 else np.nan

                chi2_df = pd.DataFrame([{
                    "n_context_used": int(table_valid.shape[0]),
                    "n_outcome_used": int(table_valid.shape[1]),
                    "N_total": n,
                    "chi2": float(chi2),
                    "df": int(dfree),
                    "p_context_outcome_assoc": float(p_ctx),
                    "cramers_v": float(cramers_v),
                    "count_mode": count_mode,
                }])
            except ValueError as e:
                chi2_df = pd.DataFrame([{
                    "n_context_used": int(table_valid.shape[0]),
                    "n_outcome_used": int(table_valid.shape[1]),
                    "N_total": float(table_valid.sum()),
                    "chi2": np.nan,
                    "df": np.nan,
                    "p_context_outcome_assoc": np.nan,
                    "cramers_v": np.nan,
                    "count_mode": count_mode,
                    "note": f"skip: {str(e)}",
                }])
        else:
            chi2_df = pd.DataFrame([{
                "n_context_used": int(table_valid.shape[0]),
                "n_outcome_used": int(table_valid.shape[1]),
                "N_total": float(table_valid.sum()) if table_valid.size else 0.0,
                "chi2": np.nan,
                "df": np.nan,
                "p_context_outcome_assoc": np.nan,
                "cramers_v": np.nan,
                "count_mode": count_mode,
                "note": "skip: table too small or empty",
            }])

    return result_df, chi2_df


In [ ]:
# =========================================================
# 3. Practice
# =========================================================
#Categorical outcome (e.g., EP_form has many labels)
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

BASE_DIR = Path(r"C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun")
CSV_PATH = BASE_DIR / "csv" / "통계용" / "V_VX_Category_2025-12-26_00-50.csv"

CONTEXT_COL="vx1_No"
OUTCOME_COL="copus" #"category"
WEIGHT_COL="ID"

CONTEXT_MIN_COUNT = 100  #변인 최소 출현량

FILTERS = {
    #"EN_label": "EF",           # EF만 사용
    #"vx1_No": lambda s: (s >= 100) & (s < 200), #상 표현 한정
    #"category": ["상상-일반", "보도·해설"],
    #"copus": 2
}
df = pd.read_csv(CSV_PATH)


# create binary ASP_No, 상 표현 마킹
df["ASP_No"] = np.where(
    df["vx1_No"].isin([101, 102, 111, 112, 123, 122, 124, 113, 125, 126, 132, 131, 114]),
    1,
    0 #상 표현 아님
)
CONTEXT_COL="ASP_No"



# Compute log-odds by context (category) and outcome (EP_form)
res, chi2 = compute_logodds_by_context_by_outcome(
    df,
    context_col=CONTEXT_COL,
    outcome_col=OUTCOME_COL,
    weight_col=WEIGHT_COL,
    filters=FILTERS,

    context_min_count=CONTEXT_MIN_COUNT,
    #outcome_top_n=30,           # keep only top 30 outcome labels
    #outcome_min_count=200,      # and/or drop rare labels
    do_chi2=False#True
)
print("\n[Preview] chi-square top 10:")
#print(chi2.head(10).to_string(index=False))



[Preview] chi-square top 10:


AttributeError: 'NoneType' object has no attribute 'head'

In [10]:
#df.info()
res.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ASP_No                   42 non-null     int64  
 1   outcome_level            42 non-null     int64  
 2   a_in_context_and_level   42 non-null     float64
 3   b_in_context_not_level   42 non-null     float64
 4   c_not_context_in_level   42 non-null     float64
 5   d_not_context_not_level  42 non-null     float64
 6   odds_ratio               42 non-null     float64
 7   log_odds                 42 non-null     float64
 8   SE                       42 non-null     float64
 9   z                        42 non-null     float64
 10  p_value                  42 non-null     float64
 11  context_total            42 non-null     float64
 12  outcome_total            42 non-null     float64
 13  grand_total              42 non-null     float64
 14  count_mode               42 

In [11]:
# =========================================================
# 4. save to file
# =========================================================

from datetime import datetime

#---- file name settings ----  
SAVE_DIR = BASE_DIR / "csv" / "결과값"
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
out_file_oz = SAVE_DIR / f'{timestamp}_logodds_{OUTCOME_COL}_by_{CONTEXT_COL}_{WEIGHT_COL}.csv'
out_file_chi2 = SAVE_DIR / f'{timestamp}_chi2_{OUTCOME_COL}_by_{CONTEXT_COL}_{WEIGHT_COL}.csv'

print(f"***저장합니다.:    {datetime.now()}***")

res.to_csv(out_file_oz, index=False, encoding="utf-8-sig")
if chi2 is not None:
    chi2.to_csv(out_file_chi2, index=False, encoding="utf-8-sig")

    print(f"[OK] saved logodds_{out_file_oz}")
    print(f"[OK] saved chi2_{out_file_chi2}")

***저장합니다.:    2025-12-27 13:59:48.480542***
[OK] saved logodds_C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\결과값\2025-12-27_13-59_logodds_copus_by_ASP_No_ID.csv
[OK] saved chi2_C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\결과값\2025-12-27_13-59_chi2_copus_by_ASP_No_ID.csv


In [ ]:
# =========================================================
#1) Boolean outcome (e.g., EP_T is 0/1 or True/False)
FILTERS = {
    "EN_label": "EF",           # EF만 사용
    "EN_No": lambda s: (s > 1100) & (s < 1102), #다, 는다 한정
    
    "copus": 2
}
res, chi2 = compute_logodds_by_context_by_outcome(
    df,
    context_col="category",
    outcome_col="EP_T",
    weight_col="ID",
    filters=FILTERS,
    outcome_top_n=None,          # not needed; only 2 levels anyway
    do_chi2=True
)
print(chi2.head(10).to_string(index=False))


In [ ]:
#3) If outcome needs normalization (strip tags, map values, etc.)

def norm_outcome(x):
    if pd.isna(x):
        return None
    return str(x).strip()

res, chi2 = compute_logodds_by_context_by_outcome(
    df,
    context_col="category",
    outcome_col="EP_form",
    outcome_transform=norm_outcome,
    weight_col="ID",
    outcome_top_n=30,
    outcome_min_count=200,
)
print(chi2.head(10).to_string(index=False))


In [10]:
import math
from typing import Tuple, Optional
from scipy import stats


def odds_stats_2x2(
    a: float, b: float, c: float, d: float,
    *,
    alpha: float = 0.5,
    add_alpha_if_zero: bool = True
) -> Tuple[float, float, float, float, float, float, float]:
    """
    Given a 2x2 table:

        outcome+   outcome-
    grp1    a         b
    grp2    c         d

    Returns:
      (odds_ratio, log_odds, SE, z, p_value, ci_low, ci_high)

    Notes:
    - If any cell is 0 and add_alpha_if_zero=True, adds alpha to ALL four cells
      (Haldane-Anscombe style smoothing).
    - Wald z-test and Wald CI are based on log(OR) ~ Normal.
    """
    if add_alpha_if_zero and (a == 0 or b == 0 or c == 0 or d == 0):
        a, b, c, d = a + alpha, b + alpha, c + alpha, d + alpha

    if b == 0 or c == 0:
        raise ZeroDivisionError("b or c is zero; set add_alpha_if_zero=True or provide nonzero cells.")

    odds_ratio = (a * d) / (b * c)
    log_odds = math.log(odds_ratio)
    se = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    z = log_odds / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))

    # 95% Wald CI on log scale, then exponentiate
    zcrit = stats.norm.ppf(0.975)
    ci_low = math.exp(log_odds - zcrit * se)
    ci_high = math.exp(log_odds + zcrit * se)

    return odds_ratio, log_odds, se, z, p_value, ci_low, ci_high



In [17]:

# -----------------------
# Example
# -----------------------
N = 710956
N_T = 319645
N_F = N - N_T
a, b = 19862,5757
c = N_T - a
d = N_F - b

if __name__ == "__main__":
    #a, b, c, d =  61188 , 82878, 710956, 40
    OR, LOR, SE, z, p, lo, hi = odds_stats_2x2(a, b, c, d)
    print(f"OR={OR:.4f}, logOR={LOR:.4f}, SE={SE:.4f}, z={z:.4f}, p={p:.4g}, 95%CI=({lo:.4f}, {hi:.4f})")


OR=4.4372, logOR=1.4900, SE=0.0152, z=98.2532, p=0, 95%CI=(4.3072, 4.5710)
